# Binding Pocket Detection

In [1]:
%load_ext autoreload
%autoreload 2

import time
from pathlib import Path
import pickle

import numpy as np
from pdbfixer import PDBFixer
from openmm.app import PDBFile
import sciapi
import scifile
import scishow
import caddpy

In [2]:
project_id = "3w32"
pdb_id = project_id
cache_dir = Path(f".tmp/{project_id}")
cache_dir.mkdir(exist_ok=True, parents=True)
pdb_filepath_raw = cache_dir / "receptor_raw.pdb"
pdb_filepath_final = cache_dir / "receptor_fixed_apo.pdb"
dogsite_pockets_filepath = cache_dir / "dogsite_pockets.pkl"
detector_filepath = cache_dir / "detector.pkl"

In [3]:
if not pdb_filepath_raw.is_file():
    pdb_file_content = sciapi.pdb.file.entry(pdb_id=pdb_id, file_format="pdb")
    pdb_filepath_raw.write_bytes(pdb_file_content)

In [4]:
if not pdb_filepath_final.is_file():
    fixer = PDBFixer(filename=str(pdb_filepath_raw))
    fixer.removeHeterogens(keepWater=False)
    fixer.addMissingHydrogens(7.0)
    PDBFile.writeFile(fixer.topology, fixer.positions, open(pdb_filepath_final, 'w'))

In [5]:
receptor = caddpy.chemsys.from_pdb(pdb_filepath_final)

In [24]:
r = caddpy.chemsys.from_pdb(pdb_filepath_final)
import nglview
n=nglview.NGLWidget()
n.add_trajectory(r)

In [25]:
n

NGLWidget()

In [6]:
if not detector_filepath.is_file():
    t_start=time.time()
    detector = caddpy.pocket.detector(receptor, gui=True, display=False, grid=0.4)
    t_end=time.time()
    print("Calculation time:", t_end - t_start)
    with open(detector_filepath, "wb") as f:
        pickle.dump(detector._detector, f)
else:
    with open(detector_filepath, "rb") as f:
        detector = caddpy.pocket.DetectorGUI(detector=pickle.load(f))

ThemeManager()

Add binding pockets detected by DoGSiteScorer (via ProteinsPlus web-API) for comparison:

In [7]:
if not dogsite_pockets_filepath.is_file():
    protplus_upload_results = sciapi.proteinsplus().upload_pdb(detector.receptor.to_pdb().to_file().encode())
    dogsite_results = sciapi.proteinsplus().dogsite(
        pdb_id=protplus_upload_results.dummy_pdb_id,
        algorithm="scorer",
    )
    dogsites = dogsite_results.full_data
    with open(dogsite_pockets_filepath, "wb") as f:
        pickle.dump(dogsites, f)
else:
    with open(dogsite_pockets_filepath, "rb") as f:
        dogsites = pickle.load(f)

for dogsite in dogsites:
    dogsite_pocket = scifile.mrc.read(dogsite["mrc"])
    detector.nglwidget.add_volume(
        dogsite_pocket.data,
        basis=dogsite_pocket.grid_vectors,
        origin=dogsite_pocket.grid_origin,
        name=f"DoG{dogsite["name"].removeprefix("P")}",
        representation_params=scishow.nglview.SurfaceRepresentationParameters(
            lazy=True, opacity=0.8, contour=True, visible=True, color=(30,30,30), isolevel=1, isolevel_type="value"
        )
    )

Display GUI:

In [21]:
detector.display()

NGLWidget(gui_style='ngl')

Accordion(children=(Output(),), titles=('Logs',))

In [10]:
detector._pockets.labels.max()

Array(7, dtype=int32)

In [23]:
receptor.trajectory.points.shape

(5119, 3)

In [19]:
np.iinfo(np.uint4).max

AttributeError: module 'numpy' has no attribute 'uint4'

In [ ]:
p0_atoms=np.array(df[df["name"]=="P_0"]["atom_serials"].to_list())
p00_atoms=np.array(df[df["name"]=="P_0_0"]["atom_serials"].to_list())

In [ ]:
for atom_name in df["name"]:
    if not atom_name.startswith("P_0_"):
        continue
    print(
        np.all(
            np.isin(
                np.array(df[df["name"]==atom_name]["atom_serials"].to_list()),
                p0_atoms,
                
            )
        )
)

In [ ]:
np.array(df[df["name"]=="P_0"]["atom_serials"].to_list())

In [20]:
import pandas as pd
df=pd.DataFrame(dogsites)
df

,name,lig_cov,poc_cov,lig_name,volume,enclosure,surface,depth,surf/vol,lid/hull,...,TYR,VAL,simpleScore,drugScore,center_x,center_y,center_z,max_radius,atom_serials,mrc
0,P_0,0.0,0.0,None,1235.20,0.10,1419.22,18.44,1.148980,-,...,1,3,0.64,0.809266,-15.65,2.80,31.72,20.56,"(5, 7, 8, 9, 12, 24, 25, 30, 32, 34, 35, 41, 7...",b'-\x00\x00\x00A\x00\x00\x00F\x00\x00\x00\x02\...
1,P_0_0,0.0,0.0,None,606.91,0.06,535.74,17.98,0.882734,-,...,0,2,0.60,0.622990,-15.14,4.50,34.59,12.10,"(323, 326, 328, 332, 418, 420, 424, 700, 702, ...","b""\x1e\x00\x00\x00(\x00\x00\x002\x00\x00\x00\x..."
2,P_0_1,0.0,0.0,None,195.52,0.08,381.97,11.38,1.953611,-,...,0,1,0.17,0.128112,-15.46,-2.94,26.94,9.65,"(1184, 1197, 1199, 1201, 1204, 1206, 1208, 121...",b'\x16\x00\x00\x00\x1c\x00\x00\x00 \x00\x00\x0...
3,P_0_2,0.0,0.0,None,190.46,0.18,291.30,9.52,1.529455,-,...,0,1,0.14,0.147133,-12.79,11.26,38.64,7.52,"(341, 342, 347, 348, 354, 356, 359, 361, 363, ...",b'\x18\x00\x00\x00\x1e\x00\x00\x00\x15\x00\x00...
4,P_0_3,0.0,0.0,None,165.25,0.09,402.33,11.61,2.434675,-,...,1,0,0.15,0.271965,-22.48,-3.98,22.87,8.44,"(5, 7, 8, 9, 12, 24, 25, 30, 32, 34, 35, 41, 7...","b'\x14\x00\x00\x00\x19\x00\x00\x00""\x00\x00\x0..."
5,P_0_4,0.0,0.0,None,77.06,0.29,166.08,7.55,2.155204,-,...,0,2,0.00,0.114479,-12.56,-2.37,23.15,5.43,"(1110, 1111, 1118, 1122, 1124, 1126, 1128, 113...",b'\x12\x00\x00\x00\x13\x00\x00\x00\x12\x00\x00...
6,P_1,0.0,0.0,None,703.81,0.13,1111.26,16.61,1.578920,-,...,1,1,0.45,0.781086,-22.45,20.61,32.52,13.39,"(363, 365, 366, 367, 766, 768, 770, 771, 772, ...","b""/\x00\x00\x00(\x00\x00\x00/\x00\x00\x00\x02\..."
7,P_1_0,0.0,0.0,None,496.38,0.12,807.28,11.12,1.626335,-,...,1,1,0.49,0.338968,-23.77,20.29,30.41,11.53,"(766, 768, 770, 771, 772, 775, 777, 785, 789, ...","b""(\x00\x00\x00(\x00\x00\x00\x1d\x00\x00\x00\x..."
8,P_1_1,0.0,0.0,None,207.42,0.17,477.51,10.57,2.302141,-,...,1,0,0.18,0.172114,-19.30,21.36,37.58,8.00,"(363, 365, 366, 367, 785, 787, 789, 790, 791, ...",b'\x1a\x00\x00\x00\x19\x00\x00\x00\x1d\x00\x00...
9,P_2,0.0,0.0,None,278.40,0.14,433.48,11.54,1.557040,-,...,1,3,0.08,0.523325,-15.26,-8.73,34.35,9.06,"(662, 670, 671, 673, 674, 675, 678, 681, 1450,...",b'!\x00\x00\x00\x19\x00\x00\x00\x19\x00\x00\x0...


In [ ]:
dog_pockets = [scifile.mrc.read(dogsite["mrc"]) for dogsite in dogsites]

In [ ]:
for dog_pocket, dog_site in zip(dog_pockets, dogsites):
    print(dog_site["name"], dog_pocket.nstart_xyz)

In [ ]:
nstarts = np.array([dog_pocket.nstart_xyz for dog_pocket in dog_pockets])
nstarts = nstarts - nstarts.min(axis=0)
nends = nstarts + np.array([dog_pocket.n_xyz for dog_pocket in dog_pockets])

In [ ]:
nstarts

In [ ]:
nends

In [ ]:
nends.max(axis=0)

In [ ]:
import numpy as np

def no_slices_overlap(starts: np.ndarray, ends: np.ndarray) -> bool:
    """
    Return True if no two axis-aligned hyper-rectangles
    defined by (starts[i], ends[i]) overlap.
    """
    # starts, ends: shape (n_slices, n_dims)
    n = starts.shape[0]

    # broadcast so that S_i[j,d] = starts[i,d],  E_j[j,d] = ends[j,d]
    S_i = starts[:, None, :]    # shape (n, n, n_dims)
    E_j = ends[None, :, :]      # shape (n, n, n_dims)
    S_j = starts[None, :, :]    # shape (n, n, n_dims)
    E_i = ends[:, None, :]      # shape (n, n, n_dims)

    # Two slices i and j overlap iff, in every dimension d:
    #      starts[i,d] < ends[j,d]  AND  starts[j,d] < ends[i,d]
    overlap_ij = np.all((S_i < E_j) & (S_j < E_i), axis=-1)

    # Clear the diagonal (a slice always “overlaps” itself)
    np.fill_diagonal(overlap_ij, False)

    # If any True remains, there is at least one overlapping pair
    return overlap_ij


In [ ]:
no_slices_overlap(nstarts, nends)